In [1]:
from sklearn.model_selection import KFold
import pandas as pd
from torch_geometric.loader import DataLoader
import lightning as L
import torch
import numpy as np

import sys
sys.path.append('../')
import FragShapley


In [2]:
# now the actual training part
classification_datasets = ['O43570', 'P00915'][:1]
n_cv = 2
n_explain = 5
expected_value = 'empty'

batch_size = 32

random_state = 42
results_folder = 'graph_classification/'

In [3]:

# load and split data
rows_performance = []
df_expl = pd.DataFrame()

for classification_dataset in classification_datasets:
    print(f'On classification dataset: {classification_dataset}')

    # load dataset
    df = pd.read_csv(f'../0_datasets/classification/{classification_dataset}.csv')

    # cross validation
    cv = KFold(n_splits=n_cv,
               shuffle=True,
               random_state=random_state,
               )
    
    for split, (train_index, test_index) in enumerate(cv.split(df)):
        print(f'\tIn split: {split}')

        # split data and get fingerprints
        smiles_train = df.nonstereo_aromatic_smiles.iloc[train_index].to_list()
        smiles_test = df.nonstereo_aromatic_smiles.iloc[test_index].to_list()
        y_train, y_test = df.label.iloc[train_index].to_list(), df.label.iloc[test_index].to_list()

        # data into dataloader
        featurizer = FragShapley.Featurizer(input_format='smiles',
                                            output_format='graph')
        ds_train = FragShapley.MoleculeDataset(list_of_inputs=smiles_train,
                                               featurizer=featurizer,
                                               y=y_train)
        ds_test = FragShapley.MoleculeDataset(list_of_inputs=smiles_test,
                                              featurizer=featurizer,
                                              y=y_test)
        dl_train = DataLoader(dataset=ds_train,
                              batch_size=batch_size,
                              )
        dl_test = DataLoader(dataset=ds_test,
                             batch_size=batch_size,
                             )
        # train using lightning
        trainer = L.Trainer(accelerator='gpu',
                            max_epochs=25,
                            )
        model = FragShapley.GCNClassifier()
        trainer.fit(model=model,
                    train_dataloaders=dl_train,
                    )
        y_pred_proba = torch.vstack(trainer.predict(model, dl_test)).detach().cpu().numpy().squeeze()
        y_pred = np.round(y_pred_proba)
        rows_performance.append({'dataset': classification_dataset,
                                 'split': split,
                                 'train_index': train_index,
                                 'test_index': test_index,
                                 'y_test': y_test,
                                 'y_pred': y_pred,
                                 'y_pred_proba': y_pred_proba,
                                 })
        
        # # now the whole explanation part
        smiles_explain = smiles_test[:n_explain]

        frag_explainer = FragShapley.FragmentExplainer(model=model,
                                                       fragmentation_method='BRICS',
                                                       representation='graph',
                                                       trainer=trainer,
                                                       featurizer=featurizer,
                                                       expected_value=expected_value,
                                                       )
        
        ev_frag = frag_explainer.expected_value
        ev_frag_logits = frag_explainer.expected_value_logit
        results_dicts, frag_to_atom_ids = frag_explainer.explain(smiles_explain)
        
        results = {'dataset': classification_dataset,
                   'split': split,
                   'smiles': smiles_explain,
                   'y_true': y_test[:n_explain],
                   'y_pred': y_pred[:n_explain],
                   'y_pred_proba': y_pred_proba[:n_explain],
                   'fragExplainer_result': results_dicts,
                   'fragExplainer_expected_value': [ev_frag for _ in smiles_explain],
                   'fragExplainer_expected_value_logits': [ev_frag_logits for _ in smiles_explain],
                   'frag_to_atom_ids': frag_to_atom_ids,
                   }
        df_expl_inner = pd.DataFrame(results)
        df_expl = pd.concat((df_expl, df_expl_inner))

df_performance = pd.DataFrame(rows_performance)
df_performance.to_pickle(results_folder + 'df_performance.pkl')
df_expl.to_pickle(results_folder + 'df_explanation.pkl')

On classification dataset: O43570
	In split: 0


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/jproth/miniconda3/envs/x10_fragShap/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name        | Type              | Params | Mode 
----------------------------

Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=25` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/jproth/miniconda3/envs/x10_fragShap/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=5` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

...

Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

In [4]:
from sklearn.metrics import balanced_accuracy_score, f1_score, matthews_corrcoef

df_performance['BACC'] = df_performance.apply(lambda x: balanced_accuracy_score(y_true=x.y_test, y_pred=x.y_pred), axis=1)
df_performance['F1'] = df_performance.apply(lambda x: f1_score(y_true=x.y_test, y_pred=x.y_pred), axis=1)
df_performance['MCC'] = df_performance.apply(lambda x: matthews_corrcoef(y_true=x.y_test, y_pred=x.y_pred), axis=1)

In [5]:
df_performance

,dataset,split,train_index,test_index,y_test,y_pred,y_pred_proba,BACC,F1,MCC
0,O43570,0,"[0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 13, 16, 18...","[8, 12, 14, 15, 17, 19, 23, 24, 25, 26, 29, 30...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.4908116, 0.49072936, 0.47269377, 0.497731, ...",0.532362,0.134199,0.164307
1,O43570,1,"[8, 12, 14, 15, 17, 19, 23, 24, 25, 26, 29, 30...","[0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 13, 16, 18...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, ...","[0.48637417, 0.4497406, 0.46667492, 0.48786598...",0.670952,0.544481,0.411790


In [6]:
def check_fragment_values_classification(fragment_dict, expected_value_logits, y_pred_proba):
    return np.allclose((sum(fragment_dict.values()) + expected_value_logits), np.log(1./(1. - y_pred_proba)))

In [7]:
df_expl['check_fragExplainer_output'] = df_expl.apply(lambda x: check_fragment_values_classification(x.fragExplainer_result, x.fragExplainer_expected_value_logits, x.y_pred_proba), axis=1)
df_expl['check_fragExplainer_output'].value_counts()

check_fragExplainer_output
True    10
Name: count, dtype: int64

In [8]:
df_expl['sum_fragExplainer_ev'] = df_expl.apply(lambda x: sum(x.fragExplainer_result.values()) + x.fragExplainer_expected_value_logits, axis=1)
df_expl['logits_predict'] = df_expl.y_pred_proba.apply(lambda x: np.log(1./(1. - x)))

In [12]:
df_expl

,dataset,split,smiles,y_true,y_pred,y_pred_proba,fragExplainer_result,fragExplainer_expected_value,fragExplainer_expected_value_logits,frag_to_atom_ids,check_fragExplainer_output,sum_fragExplainer_ev,logits_predict
0,O43570,0,C#CCNc1ccc(S(N)(=O)=O)cc1,1,0.0,0.490812,"{0: -0.018791506687800087, 1: -0.0089336136976...",0.475518,0.645341,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6, 7, 8, 9, 1...",True,0.674935,0.674937
1,O43570,0,C#CCOP(=O)(Oc1ccc(S(N)(=O)=O)cc1)c1ccccc1,1,0.0,0.490729,"{0: -0.02144002914428711, 1: 0.024022847414016...",0.475518,0.645341,"{0: [0, 1, 2], 1: [3, 4, 5, 6, 17, 18, 19, 20,...",True,0.674774,0.674776
2,O43570,0,C#CCOc1cc(N(CC)CC)ccc1C=NCc1ccc(S(N)(=O)=O)cc1,1,0.0,0.472694,"{0: -0.012414576468013576, 1: 0.02504880300589...",0.475518,0.645341,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6, 12, 13, 14...",True,0.639972,0.639974
3,O43570,0,C#CCOc1ccc(Br)cc1C=NCc1ccc(S(N)(=O)=O)cc1,1,0.0,0.497731,"{0: -0.028254007299741105, 1: 0.02960303922494...",0.475518,0.645341,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6, 7, 8, 9, 1...",True,0.688617,0.688619
4,O43570,0,C#CCOc1ccc(C=NCc2ccc(S(N)(=O)=O)cc2)cc1OC,1,0.0,0.476988,"{0: -0.018330573042233785, 1: 0.02848820586999...",0.475518,0.645341,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6, 7, 8, 9, 1...",True,0.648150,0.648151
0,O43570,1,Brc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,1,0.0,0.486374,"{0: 0.1382437547047933, 1: -0.0277049442132314...",0.470237,0.635323,"{0: [0, 1, 2, 3, 4, 25, 26], 1: [5, 6, 7, 14, ...",True,0.666258,0.666260
1,O43570,1,C#CCCCOc1ccc2ccc(=O)oc2c1,1,0.0,0.449741,"{0: -0.06667479872703552, 1: 0.087218850851058...",0.470237,0.635323,"{0: [0, 1, 2, 3, 4], 1: [5], 2: [6, 7, 8, 9, 1...",True,0.597364,0.597365
2,O43570,1,C#CCCCOc1ccc2ccc(=S)oc2c1,1,0.0,0.466675,"{0: -0.07572468121846516, 1: 0.085379312435785...",0.470237,0.635323,"{0: [0, 1, 2, 3, 4], 1: [5], 2: [6, 7, 8, 9, 1...",True,0.628622,0.628624
3,O43570,1,C#CCN(CC#C)CCCCOc1ccc(S(N)(=O)=O)cc1,1,0.0,0.487866,"{0: -0.025203793247540796, 1: -0.0124284019072...",0.470237,0.635323,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6], 3: [7, 8,...",True,0.669167,0.669169
4,O43570,1,C#CCN(CC#C)CCc1ccc(S(N)(=O)=O)cc1,1,0.0,0.495544,"{0: -0.018410955866177876, 1: -0.0086710890134...",0.470237,0.635323,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6], 3: [7, 8]...",True,0.684273,0.684275
